# Lab: Cortex Fine-Tuning

In this lab you will learn and practice the following:

❄️ Create focused training datasets for TravelBug customer support scenarios

❄️ Use SNOWFLAKE.CORTEX.FINETUNE function to create domain-specific models

❄️ Monitor fine-tuning progress and evaluate job completion

❄️ Compare base model (llama3.1-8b) vs recommended replacement model (qwen3-32b) performance

📌 **Note**:

Due to **regional workload spikes**, there may be **latency** with some of the steps. In the real world, for consistent performance, customers can explore [**Provisioned Throughput**](https://docs.snowflake.com/en/user-guide/snowflake-cortex/provisioned-throughput).

If you find your queries are running for more than 5 minutes, cancel and come back and try them later.

If you find models that are deprecated, use CoCo to help you fix the issue by selecting a suitable model.

---

### 🤖 Use CoCo as you go!

> **💡 TIP 1**: Use CoCo to explain complex SQL statements. Select any query and ask *"Explain this SQL"* to get a plain-language breakdown of what it does.
>
> **💡 TIP 2**: Want to learn more about any feature? Ask CoCo *"What does [feature name] do?"* to get more details and examples.
>
> **💡 TIP 3**: If you encounter a deprecated model error, ask CoCo *"Replace deprecated models in this notebook with current similar low-cost alternatives"* and it will fix them for you.


---

## Connect to a Service

Before running cells in this notebook, you must connect to a compute service.

**First time (create a new service):**
1. Click the **Connect** button at the top of this notebook
2. Click **Create Service** — a default name like `{{user}}_SERVICE1` will be suggested
3. Click **Service Settings** and select `ALLOW_ALL_EAI` as the external access integration
4. Leave other settings as default and click **Create**
5. Wait for the service to reach a **READY** state

**Returning (service already exists):**
1. Click the **Connect** button
2. Select your existing service from the list

Once connected, you can run Python and SQL cells interactively.

## Why Fine-Tune?


Cortex Fine-tuning is a fully managed service within Snowflake that allows you to customize popular large language models (LLMs) for specialized tasks using your own data. It employs a cost-effective method called parameter-efficient fine-tuning (PEFT) to adjust a model's behavior.

This service is an ideal middle ground when prompt engineering or retrieval augmented generation (RAG) isn't providing the desired results or latency, but training a large model from scratch is too expensive. By using your specific examples, you can improve the model's performance on domain-specific knowledge and tasks.

In the case of Travelbug, fine-tuning the model provides the following advantages.

**Domain-Specific Understanding:**
- TravelBug's unique brand voice and communication style
- Customer pain points specific to travel and adventure activities
- Context-aware responses that understand travel industry nuances

**Professional Response Quality:**
- Consistent empathy and understanding in customer interactions
- Structured information requests for efficient issue resolution
- Proper escalation procedures for serious safety or service concerns

**Business Integration:**
- Brand consistency across all automated customer interactions
- Integration with TravelBug's support policies and procedures
- Measurable improvements in customer satisfaction and resolution times

Snowflake Cortex offers a fully managed service for fine-tuning large language models (LLMs) with your own data, directly within the Snowflake platform. 

This allows you to customize popular LLMs to better suit your specific tasks and improve their performance on domain-specific queries.

The fine-tuning process is managed through a Snowflake Cortex function called **FINETUNE**, which includes commands like **CREATE**, **SHOW**, **DESCRIBE**, and **CANCEL**. 

Costs are incurred based on the number of tokens used during training and when running the AI_COMPLETE function with your fine-tuned model. 

To initiate a fine-tuning job, your role requires specific privileges, including USAGE on the database and either CREATE MODEL or OWNERSHIP on the schema.

The current model available for fine-tuning is:

*   llama3.1-8b

See [Snowflake documentation](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-finetuning) for the most up-to-date list of supported fine-tuning models, as models are periodically added and removed.


The general workflow for fine-tuning a model involves **preparing your training data**, **starting the fine-tuning job**, monitoring its progress, and then **using the resulting model for inference**. The service also provides tools for managing your fine-tuning jobs, analyzing the performance of your models, and deploying them for use.

**Cost Considerations**

*   Costs are incurred based on token usage for both training and inference. A token is the smallest text unit, roughly equal to four characters.

**Training Cost**: The cost for fine-tuning is calculated based on the number of tokens used during training. 

   **Fine-tuning trained tokens = number of input tokens * number of epochs trained**

*   **Inference Cost**: Using your fine-tuned model with the AI_COMPLETE function also incurs costs. This is based on the total tokens processed, which includes both the input prompt and the generated output.


In machine learning, an **epoch** represents one complete pass of the entire training dataset through the learning algorithm.

Think of it like studying a textbook for an exam. Reading the entire book from cover to cover one time is like completing one epoch. To learn the material thoroughly, you'd likely need to read the book multiple times. Similarly, a machine learning model usually needs to be trained for multiple epochs to learn the patterns in the data effectively.

## TravelBug Customer Support Automation

As the TravelBug customer base grows, the volume of support tickets increases, necessitating professional and empathetic responses that maintain brand consistency while effectively addressing customer concerns.

**The Challenge**:

*   Generic AI responses lack brand awareness and empathy.

*   There is an inconsistent communication style across support interactions.

*   Domain-specific knowledge about TravelBug services is missing.

*   There is no integration with business processes and escalation procedures.

**The Solution**:

**Snowflake Cortex Fine Tuning** enables the creation of an intelligent customer support agent that:

*   Maintains a consistent TravelBug brand voice and professionalism.

*   Shows appropriate empathy and understanding for customer issues.

*   Requests specific information needed for efficient issue resolution.

*   Follows proper escalation procedures for serious concerns.

### Setup your current context for the role, database, schema and warehouse.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
your_db = user + '_genai_db'
print('Your current CONTEXT information:')
print(f'Database: {your_db}')
print(f'User: {user}')
print(session)

In [ ]:
%%sql -r Setup_context_sql
USE ROLE genai_role;
USE DATABASE {{user}}_genai_db;
USE SCHEMA raw;
USE WAREHOUSE {{user}}_genai_wh;

-- Set session parameters for this lab
ALTER SESSION SET query_tag = '{{user}} lab - TOPIC: Cortex Fine-Tuning';
ALTER SESSION SET STATEMENT_TIMEOUT_IN_SECONDS = 3600;

SHOW PARAMETERS LIKE 'query_tag' IN SESSION 
  ->> SELECT "value" AS query_tag FROM $1;

## Setup and Environment Preparation

Let's begin by setting up our training data and exploring how fine-tuning can transform generic AI responses into professional, TravelBug-specific customer support interactions.

### Create focused training data for TravelBug customer support.

Now let's create a small, high-quality training dataset that teaches our model the specific communication style, empathy, and procedures that TravelBug expects from customer support interactions.

Training data key considerations:

*   **Data Source**: Your training data must originate from a Snowflake table or view.

*   **Required Columns**: When queried, your data must produce two specific columns named prompt and completion.

*   **Column Aliasing**: If your existing columns are not named prompt and completion, you must use column aliases in your query to rename them accordingly. This is a mandatory step to avoid errors.

*   **Column Content**: The prompt column should contain the input text you want to provide to the language model. The completion column should contain the corresponding ideal response that you want the model to learn.

*   **Sensitive data**: Ensure that no sensitive data is included in the training data.

In [ ]:
%%sql -r Create_focused_training_data_for_sql
--  Create training data table for TravelBug customer support fine-tuning
CREATE OR REPLACE TABLE finetuning_training_data (
    prompt TEXT,
    completion TEXT
);

-- Insert focused training examples that teach the model TravelBug's response style
INSERT INTO finetuning_training_data (prompt, completion) VALUES
-- Communication and service issues
('You are a TravelBug customer support agent. A customer says: "I''ve been trying to call your customer service for days but no one answers. I need help with my booking." Please provide a professional response.', 
 'Thank you for contacting TravelBug support. I sincerely apologize for the difficulty reaching our customer service team - this is not the level of service we strive to provide. To assist you immediately with your booking concern, please provide: 1) Your booking reference number, 2) The specific issue you need help with, and 3) Your preferred contact method. I will personally ensure you receive a response within 24 hours and escalate your feedback to improve our customer service accessibility.'),

('You are a TravelBug customer support agent. A customer says: "The guide was completely unprofessional and ruined our hiking experience. This is unacceptable." Please provide a professional response.',
 'Thank you for bringing this serious service issue to our attention. I am deeply sorry about your TravelBug hiking experience and the unprofessional conduct you encountered. We take these matters very seriously and will investigate immediately. Please provide: 1) Your booking reference number, 2) The activity date and guide details, 3) Specific details about the unprofessional behavior. I am escalating this to our operations director and you will receive a full response within 48 hours, including our action plan and appropriate compensation.'),

('You are a TravelBug customer support agent. A customer says: "The equipment you provided was broken and dangerous. I could have been seriously injured." Please provide a professional response.',
 'Thank you for reporting this critical safety issue. I am very sorry about the broken equipment and the safety risk you experienced during your TravelBug activity. Your safety is our absolute top priority. Please provide immediately: 1) Your booking reference number, 2) The specific equipment and activity details, 3) Photos of the faulty equipment if possible, 4) Whether any injuries occurred. This is being escalated to our safety team and operations director immediately for urgent investigation and corrective action.'),

('You are a TravelBug customer support agent. A customer says: "Your website had completely wrong information about the meeting point. I waited 2 hours and missed my tour." Please provide a professional response.',
 'Thank you for contacting TravelBug support. I sincerely apologize for the incorrect meeting point information on our website and the frustration of missing your tour due to our error. This is clearly our mistake and we take full responsibility. Please provide: 1) Your booking reference number, 2) The activity you booked and date, 3) Screenshot of the incorrect information if possible. We will immediately correct our website information and discuss compensation options including a full refund or rebooking at no additional cost.'),

('You are a TravelBug customer support agent. A customer says: "I need to cancel my booking for tomorrow due to a family emergency." Please provide a professional response.',
 'Thank you for contacting TravelBug support. I completely understand that family emergencies require immediate attention and flexibility. We want to help you during this difficult time. Please provide: 1) Your booking reference number, 2) The activity details for tomorrow, 3) Brief confirmation this is due to emergency circumstances. Given the emergency nature, we will waive our standard cancellation policy and process a full refund or allow you to reschedule when convenient with no additional fees.'),

('You are a TravelBug customer support agent. A customer says: "The food provided was terrible and made several people in our group sick." Please provide a professional response.',
 'Thank you for reporting this serious food safety concern. I am very sorry that the catering provided during your TravelBug experience made members of your group ill - this is completely unacceptable. We take food safety extremely seriously. Please provide: 1) Your booking reference number, 2) The activity name and date, 3) Details about the food issues and symptoms experienced, 4) Number of people affected. I am immediately escalating this to our catering partners and health safety team for urgent investigation and corrective action.');

-- Create validation data (using different scenarios)
CREATE OR REPLACE TABLE finetuning_validation_data (
    prompt TEXT,
    completion TEXT
);

INSERT INTO finetuning_validation_data (prompt, completion) VALUES
('You are a TravelBug customer support agent. A customer says: "Your staff were rude and dismissive when I raised safety concerns during the activity." Please provide a professional response.',
 'Thank you for bringing this serious service issue to our attention. I am deeply sorry about the rude and dismissive treatment you received from our staff, especially when raising legitimate safety concerns. This behavior is completely unacceptable and violates our core values. Please provide: 1) Your booking reference number, 2) The activity details and staff involved, 3) Specific details about the safety concerns and staff response. I am escalating this immediately to our operations director for urgent investigation and appropriate disciplinary action.'),

('You are a TravelBug customer support agent. A customer says: "I booked a sunset tour but it was during the middle of the day. Your schedule was completely wrong." Please provide a professional response.',
 'Thank you for contacting TravelBug support. I sincerely apologize for the incorrect scheduling information that resulted in your sunset tour occurring during midday - this is clearly our operational error. Please provide: 1) Your booking reference number, 2) The tour date and time you experienced, 3) Your booking confirmation showing the expected sunset timing. We will immediately review our scheduling systems and offer you a complimentary rebooking for an actual sunset tour or full refund for this scheduling mistake.');

### Show training data summary.

Below we can see the summary of the training data we have created.

In [ ]:
%%sql -r Show_training_data_summary_sql
SELECT 
    'Training Examples' as dataset,
    COUNT(*) as example_count,
    AVG(LENGTH(prompt)) as avg_prompt_length,
    AVG(LENGTH(completion)) as avg_completion_length
FROM finetuning_training_data
UNION ALL
SELECT 
    'Validation Examples' as dataset,
    COUNT(*) as example_count,
    AVG(LENGTH(prompt)) as avg_prompt_length,
    AVG(LENGTH(completion)) as avg_completion_length
FROM finetuning_validation_data;

## Create Fine-Tuned Model

Now let's create our fine-tuned model using the TravelBug-specific training data. This model will learn to respond with the professional, empathetic style we've defined in our training examples.

### Launch fine-tuning job.

⏰ **Important:** This process takes 5-10 minutes. Do not run all cells at once - wait for this job to complete successfully.

In [ ]:
%%sql -r Launch_finetuning_job_sql
-- Drop the model if it exists before creating. 
DROP MODEL  IF EXISTS TRAVELBUG_SUPPORT_LLAMA31_8B ;
-- Create TravelBug fine-tuned customer support model
SELECT SNOWFLAKE.CORTEX.FINETUNE(
    'CREATE',
    'TRAVELBUG_SUPPORT_LLAMA31_8B',  -- Custom model name
    'llama3.1-8b',                    -- Base model
    'SELECT prompt, completion FROM finetuning_training_data',
    'SELECT prompt, completion FROM finetuning_validation_data'
) as finetune_job_result;


In [ ]:
# Format output from the pervious cell
df = Launch_finetuning_job_sql

run_output = df.collect()[0]["FINETUNE_JOB_RESULT"]
print(run_output)

# Pass it into a Snowflake session variable
session.sql(f"SET FINETUNE_JOB_RESULT = '{run_output}';").collect()

In [ ]:
%%sql -r Launch_finetuning_job_wait_sql
-- Use the session variable in your SQL function
-- It will take some time to complete the training operation.
SELECT SNOWFLAKE.CORTEX.FINETUNE(
    'DESCRIBE',
    $FINETUNE_JOB_RESULT
) AS job_status;

In [ ]:
import json

# Convert to DataFrame
df = Launch_finetuning_job_wait_sql.to_pandas()

# Extract job status
run_output = df["JOB_STATUS"].iloc[0]

# Try formatting the output nicely
try:
    parsed_output = json.loads(run_output)
    formatted_output = json.dumps(parsed_output, indent=2)
except json.JSONDecodeError:
    # Fallback if it's not valid JSON
    formatted_output = run_output

# Display appropriate message
if "success" in run_output.lower():
    print("✅ **Your fine-tuning job was successful! You can now move to the next cell.**")
    print("### Full Job Status Output:")
    print(formatted_output)
else:
    print("⏳ **Your fine-tuning job is still in progress or did not succeed yet.** Please wait a little longer, then re-run the previous cell and this one to check the status again.")
    print("### Current Job Status Output:")
    print(formatted_output)

### Monitor Fine-Tuning Job Status & Costs

Use `SNOWFLAKE.CORTEX.FINETUNE('DESCRIBE', ...)` to check job progress, and query the `CORTEX_FINE_TUNING_USAGE_HISTORY` view for credit/token consumption.

In [ ]:
%%sql -r all_finetune_jobs_sql
-- Show all fine-tuning jobs in this account
SELECT SNOWFLAKE.CORTEX.FINETUNE('SHOW') AS all_jobs;

In [ ]:
%%sql -r finetune_job_details_sql
-- Detailed status of the current fine-tuning job
SELECT SNOWFLAKE.CORTEX.FINETUNE(
    'DESCRIBE',
    $FINETUNE_JOB_RESULT
) AS job_details;

In [ ]:
%%sql -r finetune_usage_history_sql
-- Fine-tuning credit and token usage history
SELECT *
  FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_FINE_TUNING_USAGE_HISTORY
  ORDER BY START_TIME DESC
  LIMIT 20;

## Compare Base Model vs Recommended Replacement Model

Now for the exciting part! Let's test realistic customer scenarios and compare responses from the base model (llama3.1-8b) against the recommended replacement model (qwen3-32b) to see how model choice affects response quality.


### Test realistic TravelBug customer scenarios.

Let's see how both models handle real customer support situations that TravelBug might encounter. In order to see the results you will need to click on the **View Display Options** for the cell and click **Results Only**

In [ ]:
%%sql -r Test_realistic_TravelBug_customer_sql
WITH test_scenarios AS (
    -- Use the VALUES clause to create an in-memory table with our test cases.
    SELECT scenario_id, customer_issue, expected_focus FROM VALUES
        -- Each row represents a unique customer support scenario.
        -- Columns: scenario_id (unique identifier), customer_issue (the customer's message),
        -- and expected_focus (the key theme or category of the issue).
        (1, 'My hiking boots were completely soaked through during the mountain trek. This ruined my entire experience.', 'equipment_quality'),
        (2, 'I booked a snorkelling trip but the weather was terrible. Can I get a refund?', 'weather_refund'),
        (3, 'The guide didnt speak English very well and I missed important safety instructions.', 'communication_safety'),
        (4, 'Je voudrais annuler ma réservation pour demain à cause dune urgence familiale.', 'emergency_cancellation'), -- French language test case
        (5, 'The meeting point was completely wrong on your website. I waited for 2 hours!', 'logistics_error'),
        (6, 'Your customer service phone number just rings busy all day. How am I supposed to get help?', 'access_issues'),
        (7, 'The activity was canceled last minute but nobody told us. We wasted our whole day.', 'communication_failure'),
        (8, 'Food poisoning from your catered lunch. Three people in our group got sick.', 'health_safety')
    -- Alias the created table and its columns for clarity.
    as test_data(scenario_id, customer_issue, expected_focus)
),
-- Define a second CTE named 'responses' to generate AI responses for each test scenario.
responses AS (
    SELECT
        -- Pass through the columns from the 'test_scenarios' CTE.
        scenario_id,
        customer_issue,
        expected_focus,

        -- Call the Snowflake Cortex 'COMPLETE' function to get a response from the base model.
        -- This serves as our baseline or control group.
        SNOWFLAKE.CORTEX.COMPLETE(
            'llama3.1-8b', -- Specify the generic, pre-trained model to use.
            -- Construct the prompt by embedding the customer issue into a template.
            'You are a customer support agent. A customer says: "' || customer_issue || '". Provide a helpful response.'
        ) as base_response, -- Alias the output column for the base model's response.

        -- Call the Snowflake Cortex 'COMPLETE' function again, this time for the recommended replacement model.
        -- qwen3-32b is the recommended replacement for deprecated fine-tuned model inference.
        SNOWFLAKE.CORTEX.COMPLETE(
            'qwen3-32b', -- Specify the recommended replacement model.
            -- Construct a similar prompt, but tailored for a 'TravelBug' agent.
            'You are a TravelBug customer support agent. A customer says: "' || customer_issue || '". Please provide a professional response.'
        ) as replacement_response -- Alias the output column for the replacement model's response.

    FROM test_scenarios -- Source the data from our 'test_scenarios' CTE.
)
-- Final SELECT statement to format and display the results for comparison.
SELECT
    scenario_id,        -- The unique ID for the test case.
    expected_focus,     -- The category of the issue.
    -- Format the original customer issue by wrapping it in double quotes for readability.
    '"' || customer_issue || '"' as customer_issue,
    -- Format the base model's response by adding a clear header.
    '\n--- BASE MODEL RESPONSE ---\n' || base_response as base_model_response,
    -- Format the replacement model's response by adding its own clear header.
    '\n--- REPLACEMENT MODEL RESPONSE ---\n' || replacement_response as replacement_model_response
FROM responses -- Select the data from the 'responses' CTE where the AI generation occurred.
ORDER BY scenario_id; -- Order the final output by the scenario ID for a consistent and logical view.


### Evaluation of Model Performance: Base vs Recommended Replacement

By running the query below, we were able to systematically compare the responses generated by the **base model(llama3.1-8b)** and the **recommended replacement model (qwen3-32b)** across a set of realistic customer service scenarios. The analysis focused on five key metrics that reflect both response quality and business relevance.

**Key Metrics Compared**:

Run the query to see how llama3.1-8b (base) and qwen3-32b (recommended replacement) compare across these five metrics: Brand Mention Rate, Asks for Booking Info, Shows Empathy, Escalates Issues, and Average Response Length. Results will vary based on model versions and prompts.


**Analyze the response quality differences**

Let's quantify the differences between the base model and replacement model responses.

The following query is designed to quantitatively measure the performance improvement of the **recommended replacement model (qwen3-32b)** compared to the **base model (llama3.1-8b)** for generating customer support responses.

It operates in three main stages:

*   **Test Scenarios**: It establishes a standardized set of eight common customer support issues to use as a consistent test bed.

*   **Response Generation & Analysis**: For each scenario, it prompts both the base model and the recommended replacement model to generate a response. It then analyzes each response for specific, desirable characteristics such as mentioning the brand name, showing empathy, asking for booking information, and escalating the issue when necessary.

*   **Metric Aggregation**: Finally, it aggregates the results to calculate and compare the average performance of each model across these key quality metrics, presenting the final output as a summary table that clearly shows the performance comparison between the two models.

In [ ]:
%%sql -r Evaluation_of_FineTuned_model_sql
WITH test_scenarios AS (
    SELECT scenario_id, customer_issue, expected_focus FROM VALUES 
        (1, 'My hiking boots were completely soaked through during the mountain trek. This ruined my entire experience.', 'equipment_quality'),
        (2, 'I booked a snorkelling trip but the weather was terrible. Can I get a refund?', 'weather_refund'),
        (3, 'The guide didnt speak English very well and I missed important safety instructions.', 'communication_safety'),
        (4, 'Je voudrais annuler ma réservation pour demain à cause dune urgence familiale.', 'emergency_cancellation'),
        (5, 'The meeting point was completely wrong on your website. I waited for 2 hours!', 'logistics_error'),
        (6, 'Your customer service phone number just rings busy all day. How am I supposed to get help?', 'access_issues'),
        (7, 'The activity was canceled last minute but nobody told us. We wasted our whole day.', 'communication_failure'),
        (8, 'Food poisoning from your catered lunch. Three people in our group got sick.', 'health_safety')
    as test_data(scenario_id, customer_issue, expected_focus)
),
response_analysis AS (
    SELECT 
        scenario_id,
        customer_issue,
        expected_focus,
        
        SNOWFLAKE.CORTEX.COMPLETE(
            'llama3.1-8b',
            'You are a customer support agent. A customer says: "' || customer_issue || '". Provide a helpful response.'
        ) as base_response,
        
        SNOWFLAKE.CORTEX.COMPLETE(
            'qwen3-32b',
            'You are a TravelBug customer support agent. A customer says: "' || customer_issue || '". Please provide a professional response.'
        ) as replacement_response
        
    FROM test_scenarios
),
quality_metrics AS (
    SELECT
        scenario_id,
        expected_focus,
        
        -- Analyze base model characteristics
        LENGTH(base_response) as base_response_length,
        CASE WHEN UPPER(base_response) LIKE '%TRAVELBUG%' THEN 1 ELSE 0 END as base_mentions_brand,
        CASE WHEN UPPER(base_response) LIKE '%BOOKING REFERENCE%' OR UPPER(base_response) LIKE '%BOOKING%' THEN 1 ELSE 0 END as base_asks_booking_info,
        CASE WHEN UPPER(base_response) LIKE '%APOLOGIZE%' OR UPPER(base_response) LIKE '%SORRY%' THEN 1 ELSE 0 END as base_shows_empathy,
        CASE WHEN UPPER(base_response) LIKE '%ESCALAT%' OR UPPER(base_response) LIKE '%INVESTIGATE%' THEN 1 ELSE 0 END as base_escalates,
        
        -- Analyze replacement model (qwen3-32b) characteristics
        LENGTH(replacement_response) as replacement_response_length,
        CASE WHEN UPPER(replacement_response) LIKE '%TRAVELBUG%' THEN 1 ELSE 0 END as replacement_mentions_brand,
        CASE WHEN UPPER(replacement_response) LIKE '%BOOKING REFERENCE%' OR UPPER(replacement_response) LIKE '%BOOKING%' THEN 1 ELSE 0 END as replacement_asks_booking_info,
        CASE WHEN UPPER(replacement_response) LIKE '%APOLOGIZE%' OR UPPER(replacement_response) LIKE '%SORRY%' THEN 1 ELSE 0 END as replacement_shows_empathy,
        CASE WHEN UPPER(replacement_response) LIKE '%ESCALAT%' OR UPPER(replacement_response) LIKE '%INVESTIGATE%' THEN 1 ELSE 0 END as replacement_escalates
        
    FROM response_analysis
)
SELECT 
    'Average Response Length (characters)' as metric,
    ROUND(AVG(base_response_length), 0) as base_model_avg,
    ROUND(AVG(replacement_response_length), 0) as replacement_model_avg,
    ROUND(AVG(replacement_response_length) - AVG(base_response_length), 0) as improvement
FROM quality_metrics

UNION ALL

SELECT 
    'Brand Mention Rate (%)' as metric,
    ROUND(AVG(base_mentions_brand) * 100, 1) as base_model_avg,
    ROUND(AVG(replacement_mentions_brand) * 100, 1) as replacement_model_avg,
    ROUND((AVG(replacement_mentions_brand) - AVG(base_mentions_brand)) * 100, 1) as improvement
FROM quality_metrics

UNION ALL

SELECT 
    'Asks for Booking Info (%)' as metric,
    ROUND(AVG(base_asks_booking_info) * 100, 1) as base_model_avg,
    ROUND(AVG(replacement_asks_booking_info) * 100, 1) as replacement_model_avg,
    ROUND((AVG(replacement_asks_booking_info) - AVG(base_asks_booking_info)) * 100, 1) as improvement
FROM quality_metrics

UNION ALL

SELECT 
    'Shows Empathy (%)' as metric,
    ROUND(AVG(base_shows_empathy) * 100, 1) as base_model_avg,
    ROUND(AVG(replacement_shows_empathy) * 100, 1) as replacement_model_avg,
    ROUND((AVG(replacement_shows_empathy) - AVG(base_shows_empathy)) * 100, 1) as improvement
FROM quality_metrics

UNION ALL

SELECT 
    'Escalates Issues (%)' as metric,
    ROUND(AVG(base_escalates) * 100, 1) as base_model_avg,
    ROUND(AVG(replacement_escalates) * 100, 1) as replacement_model_avg,
    ROUND((AVG(replacement_escalates) - AVG(base_escalates)) * 100, 1) as improvement
FROM quality_metrics;

## 🎯 Challenge Questions

Test your understanding of the concepts covered in this lab.

In [ ]:
from snowflake.snowpark.context import get_active_session
from IPython.display import display, HTML

session = get_active_session()

quiz_data = [
    {"q": "What is the primary purpose of fine-tuning an LLM?", "options": ["A) To make the model run faster", "B) Fine-tuning customizes a base model with domain-specific training data for better task performance", "C) To reduce the cost of model inference", "D) To increase the model's parameter count"], "hash": "ec59e47ee17606f64076bef29c698cd1"},
    {"q": "What should training data for fine-tuning include?", "options": ["A) Only questions without answers", "B) Random text samples from the internet", "C) Training data should include input-output pairs that demonstrate the desired behavior", "D) Only the model's original training data"], "hash": "78456b94f8537ba1ad7e835d7a891875"},
    {"q": "What is a key benefit of using a fine-tuned model over a base model?", "options": ["A) Fine-tuned models can provide more consistent and domain-appropriate responses", "B) Fine-tuned models are always faster", "C) Fine-tuned models require no prompting", "D) Fine-tuned models never make mistakes"], "hash": "c1badd7157aa79136b73b1746fd8081d"},
    {"q": "How does Snowflake Cortex handle the fine-tuning process?", "options": ["A) Users must provision their own GPU servers", "B) Fine-tuning requires manual model deployment", "C) Users must manage the training loop themselves", "D) Fine-tuning is a managed service that handles the training infrastructure automatically"], "hash": "35b0d590fcb01699172efbb3d27583e7"},
    {"q": "Why is it important to compare base and fine-tuned model outputs?", "options": ["A) To ensure the base model is deleted", "B) Comparing base and fine-tuned model outputs helps evaluate the improvement in response quality", "C) To verify the models use the same weights", "D) To reduce compute costs"], "hash": "9f74b1342e866fd2779f2af8280eae84"},
]

results_map = {}
for qi, item in enumerate(quiz_data):
    results_map[qi] = {}
    for opt in item["options"]:
        letter = opt[0]
        escaped_opt = opt.replace("'", "''")
        result = session.sql(f"CALL genai_db.resources.quiz_temp('{item['hash']}', '{escaped_opt}', 'False')").collect()
        feedback = result[0][0]
        is_correct = 'Correct' in feedback or '✅' in feedback
        results_map[qi][letter] = (feedback, is_correct)

html = """<style>
.cq { margin: 20px 0; padding: 16px; border: 1px solid #d0d0d0; border-radius: 10px; background: #fafafa; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif; font-size: 14px; }
.cq h4 { font-family: inherit; }
.cq input[type="radio"] { display: none; }
.cq .lbl { display: block; padding: 8px 12px; border-radius: 6px; cursor: pointer; font-family: inherit; }
.cq .lbl:hover { background: #e8f0fe; }
.cq input[type="radio"]:checked + .lbl { border-color: #1a73e8; background: #e8f0fe; font-weight: 600; }
.cq .fb { display: none; padding: 6px 12px; margin-top: 2px; border-radius: 4px; font-weight: 600; font-family: inherit; }
.cq input[type="radio"]:checked + .lbl + .fb { display: block; }
.cq .fb.ok { background: #e6f4ea; color: #1e7e34; }
.cq .fb.no { background: #fce8e6; color: #c62828; }
</style>"""

for qi, item in enumerate(quiz_data):
    html += f'<div class="cq"><h4>Q{qi+1}: {item["q"]}</h4>'
    for opt in item["options"]:
        letter = opt[0]
        feedback, is_correct = results_map[qi][letter]
        css_class = 'ok' if is_correct else 'no'
        uid = f'cq{qi}_{letter}'
        html += f'<div class="opt"><input type="radio" name="cq{qi}" id="{uid}">'
        html += f'<label class="lbl" for="{uid}">{opt}</label>'
        html += f'<div class="fb {css_class}">{letter}) {feedback}</div></div>'
    html += '</div>'

display(HTML(html))

## Key Takeaways

❄️ Fine-Tuning Transforms Generic AI into Domain Experts

❄️ Training data requires `prompt` and `completion` columns with example input-output pairs

❄️ Can monitor the progress of a model when it is created

❄️ It is possible to gain clear improvements over base model responses  

❄️ Models are persisted and can be viewed using SQL or AI&ML Models screen